<a href="https://colab.research.google.com/github/SriSharanya-617/GPT/blob/main/TinyStories_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

In [35]:
print(ds["train"][0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [36]:
train_texts = ds["train"]["text"][:750]  # use subset initially
print("Number of stories:", len(train_texts))

Number of stories: 750


Clean Text

In [37]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_texts = [clean_text(t) for t in train_texts]

Task 2: Tokenization

In [38]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    oov_token="<unk>"
)

tokenizer.fit_on_texts(train_texts)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 4285


In [39]:
MAX_TOKENS = 30

sequences = [
    tokenizer.texts_to_sequences([t])[0][:MAX_TOKENS]
    for t in train_texts
]
print(sequences[0][:20])

[26, 19, 5, 22, 31, 87, 43, 89, 5, 1281, 13, 10, 178, 8, 145, 9, 6, 1408, 4, 50]


Task 3: Create GPT Training Sequences

*   once upon a time there was a rabbit

Input: once
Target: upon

Input: once upon
Target: a

Input: once upon a
Target: time



In [40]:
training_sequences = []

for seq in sequences:
    for i in range(1, len(seq)):
        training_sequences.append(
            (seq[:i], seq[i])
        )

Task 4: Padding

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

X = []
y = []

for inp, target in training_sequences:
    X.append(inp)
    y.append(target)

max_len = max(len(seq) for seq in X)

X = pad_sequences(
    X,
    maxlen=max_len,
    padding='pre'
)

y = np.array(y)

print(X.shape)
print(y.shape)

(21750, 29)
(21750,)


Task 5: Embedding Layer

In [42]:
import tensorflow as tf

In [43]:
embedding_dim = 64

embedding = tf.keras.layers.Embedding(
    vocab_size,
    embedding_dim
)

Task 6: Positional Encoding

In [44]:
import numpy as np

def positional_encoding(max_len, d_model):

    pos = np.arange(max_len)[:, np.newaxis]

    i = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (i // 2)) / np.float32(d_model)
    )

    angles = pos * angle_rates

    pe = np.zeros((max_len, d_model))

    pe[:, 0::2] = np.sin(angles[:, 0::2])

    pe[:, 1::2] = np.cos(angles[:, 1::2])

    return pe

Task 7: Masked Attention

In [45]:
seq_len = 20

mask = tf.linalg.band_part(
    tf.ones((seq_len, seq_len)),
    -1,
    0
)

Task 8: GPT Decoder Block

In [46]:
class DecoderBlock(tf.keras.layers.Layer):

    def __init__(self, embed_dim, num_heads, ff_dim):

        super().__init__()

        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(
                ff_dim,
                activation='relu'
            ),
            tf.keras.layers.Dense(embed_dim)
        ])

        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x):

        attn_output = self.att(
            x,
            x,
            use_causal_mask=True
        )

        x = self.norm1(x + attn_output)

        ffn_output = self.ffn(x)

        x = self.norm2(x + ffn_output)

        return x

Task 9: Build GPT Model

In [47]:
inputs = tf.keras.Input(
    shape=(max_len,)
)

x = tf.keras.layers.Embedding(
    vocab_size,
    64
)(inputs)

x = DecoderBlock(
    64,
    2,
    128
)(x)



x = tf.keras.layers.Lambda(
    lambda t: t[:, -1, :]
)(x)

outputs = tf.keras.layers.Dense(
    vocab_size,
    activation='softmax'
)(x)

model = tf.keras.Model(
    inputs,
    outputs
)

Task 10: Compile & Train

In [48]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X,
    y,
    batch_size=64,
    epochs=10,
    validation_split=0.1
)

Epoch 1/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 27s 76ms/step - accuracy: 0.2305 - loss: 5.0227 - val_accuracy: 0.2570 - val_loss: 4.7400
Epoch 2/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 23s 75ms/step - accuracy: 0.3470 - loss: 3.6277 - val_accuracy: 0.2708 - val_loss: 4.5364
Epoch 3/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 23s 74ms/step - accuracy: 0.3896 - loss: 3.1645 - val_accuracy: 0.2947 - val_loss: 4.5728
Epoch 4/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 21s 69ms/step - accuracy: 0.4156 - loss: 2.8635 - val_accuracy: 0.2749 - val_loss: 4.7256
Epoch 5/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 23s 74ms/step - accuracy: 0.4351 - loss: 2.6330 - val_accuracy: 0.2653 - val_loss: 4.7864
Epoch 6/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 23s 77ms/step - accuracy: 0.4509 - loss: 2.4435 - val_accuracy: 0.2828 - val_loss: 4.8696
Epoch 7/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 23s 74ms/step - accuracy: 0.4686 - loss: 2.2719 - val_accuracy: 0.2869 - val_loss: 4.9740
Epoch 8/10
306/306 ━━━━━━━━━━━━━━━━━━━━ 21s 70ms/step - accuracy: 0.4874 - loss: 2.1184 - 

Task 11: Story Generation

In [49]:
import numpy as np

prompt = "once upon a time"

for _ in range(30):

    tokens = tokenizer.texts_to_sequences(
        [prompt]
    )[0]

    padded = pad_sequences(
        [tokens],
        maxlen=max_len,
        padding='pre'
    )

    pred = model.predict(
        padded,
        verbose=0
    )

    next_id = np.argmax(pred[0])

    next_word = tokenizer.index_word.get(
        next_id,
        ""
    )

    prompt += " " + next_word

print(prompt)

once upon a time there was a little girl named molly molly had a very special trip and her mom had a great adventure her mom had a big idea she had lots of


Task 12: Save Model

In [50]:
model.save("story_gpt.keras")

In [51]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [52]:
import numpy as np

# Evaluate model
loss, accuracy = model.evaluate(X, y, verbose=0)

# Calculate perplexity
perplexity = np.exp(loss)

print("Loss:", loss)
print("Accuracy:", accuracy)
print("Perplexity:", perplexity)

Loss: 1.9131073951721191
Accuracy: 0.5735632181167603
Perplexity: 6.774105951448855


In [53]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

prompt = "once upon a time"

for _ in range(20):

    tokens = tokenizer.texts_to_sequences([prompt])[0]

    padded = pad_sequences(
        [tokens],
        maxlen=max_len,
        padding='pre'
    )

    pred = model.predict(padded, verbose=0)

    next_id = np.argmax(pred[0])

    next_word = tokenizer.index_word.get(next_id, "")

    prompt += " " + next_word

print("Generated Story:")
print(prompt)

Generated Story:
once upon a time there was a little girl named molly molly had a very special trip and her mom had a great adventure


In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

perplexity = np.exp(loss)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)
print("Test Perplexity:", perplexity)

Test Loss: 1.9191443920135498
Test Accuracy: 0.569655179977417
Test Perplexity: 6.81512489872984
